# 特征提取：从原始数据中构造新特征
> 难度标签：中级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：11_特征工程 | 源文件：特征提取.py | 核心功能：PCA、LDA、特征变换

## 概述

特征提取将原始特征变换为更有信息量的新特征。PCA（无监督）和 LDA（有监督）是最常用的方法。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_classification

## 数学原理

### 1. PCA 主成分分析

PCA 寻找方差最大的投影方向。设数据矩阵 $X \in \mathbb{R}^{n \times d}$（已中心化），协方差矩阵：

$$\Sigma = \frac{1}{n}X^\top X$$

对 $\Sigma$ 做特征分解：$\Sigma = V \Lambda V^\top$，选择前 $k$ 个最大特征值对应的特征向量。

投影：$Z = X V_k$，其中 $V_k \in \mathbb{R}^{d \times k}$。

**方差贡献率**：第 $j$ 个主成分解释的方差比例

$$\eta_j = \frac{\lambda_j}{\sum_{i=1}^{d}\lambda_i}$$

**累计贡献率**：前 $k$ 个主成分解释的总方差比例

$$\eta_{cum} = \frac{\sum_{j=1}^{k}\lambda_j}{\sum_{i=1}^{d}\lambda_i}$$

代码中 `PCA(n_components=0.95)` 保留累计贡献率 $\geq 95\%$ 的主成分数。

### 2. 多项式特征（PolynomialFeatures）

对原始特征 $[x_1, x_2]$，degree=2 生成：

$$[x_1, x_2, x_1^2, x_1 x_2, x_2^2]$$

一般地，$d$ 个特征 degree=2 生成 $\binom{d+2}{2}$ 个特征（含偏置）。

`interaction_only=True` 只生成交叉项：$[x_1, x_2, x_1 x_2]$，不含平方项。

### 3. TF-IDF 文本特征提取

将文本转换为数值向量（详见 08_NLP/TF_IDF.md）：

$$\text{TF-IDF}(t, d) = \frac{c(t,d)}{\sum_{t'}c(t',d)} \times \log\frac{N}{|\{d: t \in d\}|}$$

### 4. 图像块特征

`PatchExtractor` 从图像中提取固定大小的像素块，展平为特征向量。

### 5. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `PCA(n_components=0.95)` | 保留 $\eta_{cum} \geq 0.95$ 的主成分 |
| `pca.explained_variance_ratio_` | $\eta_j = \lambda_j / \sum \lambda_i$ |
| `PCA(n_components=3)` | 投影到前 3 个主成分：$Z = XV_3$ |
| `PolynomialFeatures(degree=2)` | 生成 $\{x_1, x_2, x_1^2, x_1x_2, x_2^2\}$ |
| `interaction_only=True` | 只生成交叉项 $x_1 x_2$ |
| `TfidfVectorizer()` | 文本 $\to$ TF-IDF 矩阵 |

### 1. PCA 主成分分析

运行 1. PCA 主成分分析 的代码，观察算法在该环节的行为。

In [ ]:
from sklearn.decomposition import PCA

print("=" * 60)
print("1. PCA 主成分分析")
print("=" * 60)

# 生成高维数据（10个特征）
X_pca, y_pca = make_classification(
    n_samples=200, n_features=10, n_informative=3,
    n_redundant=2, random_state=42
)

print(f"原始数据维度: {X_pca.shape}")

# 保留主成分能解释95%方差
pca_auto = PCA(n_components=0.95, random_state=42)
X_reduced = pca_auto.fit_transform(X_pca)

print(f"保留95%方差后维度: {X_reduced.shape}")
print(f"各主成分方差贡献率: {np.round(pca_auto.explained_variance_ratio_, 4)}")
print(f"累计方差贡献率:     {np.round(np.cumsum(pca_auto.explained_variance_ratio_), 4)}")
print(f"主成分数量: {pca_auto.n_components_} / 原始: {X_pca.shape[1]}")

# 指定保留主成分数量
pca_3 = PCA(n_components=3, random_state=42)
X_3d = pca_3.fit_transform(X_pca)
print(f"\n指定保留3个主成分: {X_3d.shape}")
print(f"3个主成分方差贡献率: {np.round(pca_3.explained_variance_ratio_, 4)}")
print(f"3个主成分累计贡献率: {pca_3.explained_variance_ratio_.sum():.4f}")

### 2. 多项式特征 (PolynomialFeatures)

运行 2. 多项式特征 (PolynomialFeatures) 的代码，观察算法在该环节的行为。

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

print("\n" + "=" * 60)
print("2. 多项式特征 (PolynomialFeatures)")
print("=" * 60)

# 简单的2特征示例，便于理解
X_poly_demo = np.array([[2, 3],
                         [4, 5],
                         [6, 1]])
print(f"原始数据 (2个特征):\n{X_poly_demo}")

# degree=2, interaction_only=False: 包含平方项和交叉项
poly_full = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_poly_full = poly_full.fit_transform(X_poly_demo)
print(f"\ndegree=2 (含平方项和交叉项):")
print(f"特征名: {poly_full.get_feature_names_out(['x1', 'x2'])}")
print(f"形状: {X_poly_full.shape}")
# --- 输出结果 ---
print(f"数据:\n{X_poly_full}")

# interaction_only=True: 只有交叉项，无平方项
poly_interact = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
X_poly_inter = poly_interact.fit_transform(X_poly_demo)
print(f"\ninteraction_only=True (仅交叉项):")
print(f"特征名: {poly_interact.get_feature_names_out(['x1', 'x2'])}")
print(f"数据:\n{X_poly_inter}")

### 3. 文本特征 (TfidfVectorizer)

运行 3. 文本特征 (TfidfVectorizer) 的代码，观察算法在该环节的行为。

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("\n" + "=" * 60)
print("3. 文本特征 (TfidfVectorizer)")
print("=" * 60)

# 示例文档集
corpus = [
    "机器学习是人工智能的一个分支",
    "深度学习使用神经网络模型",
    "支持向量机是一种监督学习算法",
    "自然语言处理用于分析文本数据",
# --- 继续 ---
    "卷积神经网络在图像识别中效果很好",
]

print("原始文档:")
for i, doc in enumerate(corpus):
    print(f"  [{i}] {doc}")

# TF-IDF向量化
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)

print(f"\nTF-IDF矩阵形状: {X_tfidf.shape} (文档数 x 词汇量)")
print(f"词汇表: {tfidf.get_feature_names_out()[:10]}... (共{len(tfidf.get_feature_names_out())}个)")

# 以稀疏矩阵形式查看
X_dense = X_tfidf.toarray()
print(f"\nTF-IDF矩阵 (稠密形式，保留3位小数):")
print(f"{'词汇':<12}", end="")
for i in range(min(8, X_dense.shape[1])):
    print(f"{tfidf.get_feature_names_out()[i]:<10}", end="")
# --- 输出结果 ---
print("...")
for doc_idx in range(len(corpus)):
    print(f"文档{doc_idx}:  ", end="")
    for feat_idx in range(min(8, X_dense.shape[1])):
        print(f"{X_dense[doc_idx, feat_idx]:<10.3f}", end="")
# --- 输出结果 ---
    print("...")

# IDF权重：衡量词的区分能力
print("\nIDF权重 (高=更独特):")
idf = tfidf.idf_
top_idf_idx = np.argsort(idf)[::-1][:8]
for idx in top_idf_idx:
    word = tfidf.get_feature_names_out()[idx]
# --- 输出结果 ---
    print(f"  {word}: {idf[idx]:.4f}")

### 4. 图像块特征 (PatchExtractor)

运行 4. 图像块特征 (PatchExtractor) 的代码，观察算法在该环节的行为。

In [ ]:
from sklearn.feature_extraction.image import PatchExtractor

print("\n" + "=" * 60)
print("4. 图像块特征 (PatchExtractor)")
print("=" * 60)

# 模拟一张灰度图像 (28x28)
np.random.seed(42)
fake_image = np.random.rand(1, 28, 28)
print(f"模拟图像形状: {fake_image.shape} (1张, 28x28像素)")

# 提取8x8的图像块
patch_extractor = PatchExtractor(patch_size=(8, 8), random_state=42)
patches = patch_extractor.transform(fake_image)

print(f"提取的块数量: {patches.shape[0]}")
print(f"每个块的大小: {patches.shape[1]}x{patches.shape[2]}")
print(f"展平后每个块的特征维度: {patches.shape[1] * patches.shape[2]}")

# 查看一个块的统计信息
patch_0 = patches[0]
print(f"\n第1个块的统计:")
print(f"  最小值: {patch_0.min():.4f}, 最大值: {patch_0.max():.4f}")
print(f"  均值: {patch_0.mean():.4f}, 标准差: {patch_0.std():.4f}")

# 多张图像的块提取
images = np.random.rand(5, 32, 32)
patches_multi = patch_extractor.transform(images)
print(f"\n5张32x32图像提取8x8块后:")
print(f"  总块数: {patches_multi.shape[0]}")
print(f"  每个块: {patches_multi.shape[1]}x{patches_multi.shape[2]}")

### 综合对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n" + "=" * 60)
print("综合对比：特征提取方法")
print("=" * 60)
print(f"PCA:           {X_pca.shape[1]}维 -> {X_reduced.shape[1]}维 (保留95%方差)")
print(f"多项式特征:    2维 -> {X_poly_full.shape[1]}维 (degree=2)")
# --- 输出结果 ---
print(f"TF-IDF文本:    {len(corpus)}篇文档 -> {X_tfidf.shape[1]}维词汇空间")
print(f"图像块特征:    28x28 -> {patches.shape[0]}个8x8块")

## 关键代码解释

```python
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)
X_new = pca.fit_transform(X)
```

## 注意事项

1. 提取后特征失去了原始可解释性
2. 必须在训练集上 fit
3. 非线性提取（t-SNE、UMAP）不能 transform 新数据

## 总结与延伸

以上代码展示了 **特征提取** 的完整流程。

**解读要点**：
- 关注 **特征重要性** 排名和分布
- 对比特征选择前后的模型性能
- 观察特征交叉是否带来性能提升

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **自动特征工程**：Featuretools 等工具
- **深度特征提取**：用预训练网络提取特征
- **时间序列特征提取**：tsfresh 库
